Group 120 – Assignment 1 – PS4

| Name                       | Roll Number | Contribution Percentage |
| -------------------------- | ----------- | ----------------------- |
| BOLLAM MANINDRA            | 2024AA05733 | 20%                     |
| ASHVINI SATISH BHOSALE     | 2024AA05732 | 20%                     |
| KUMARI POONAM SURESH NEETA | 2024AA05075 | 20%                     |
| ABHIRAM BORIGE             | 2024AA05909 | 20%                     |
| SASANE GAURI RAMESH        | 2024AA05679 | 20%                     |


In [ ]:
import os
import platform
import time
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms

In [3]:
# ---------------------
# Model
# ---------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

In [4]:
# ---------------------
# Distributed init
# ---------------------

def select_backend() -> str:
    on_windows = platform.system().lower().startswith("win")
    nccl_ok = hasattr(dist, "is_nccl_available") and dist.is_nccl_available()
    if torch.cuda.is_available() and nccl_ok and not on_windows:
        return "nccl"
    return "gloo"


def maybe_init_dist() -> Tuple[int, int, int, str]:
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if world_size > 1 and not dist.is_initialized():
        backend = select_backend()
        dist.init_process_group(backend=backend)
    else:
        backend = "none"
    return rank, world_size, local_rank, backend

In [5]:
# ---------------------
# Data
# ---------------------

def build_loaders(batch_size: int, world_size: int, rank: int, num_workers: int):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(root="data", train=False, download=True, transform=transform)

    if world_size > 1:
        train_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
        test_sampler = DistributedSampler(test_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)
    else:
        train_sampler = None
        test_sampler = None

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        sampler=test_sampler,
        num_workers=max(0, num_workers // 2),
        pin_memory=True,
        persistent_workers=(num_workers // 2) > 0,
    )
    return train_loader, test_loader, train_sampler


In [6]:

# ---------------------
# Train / Eval
# ---------------------

def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / max(1, total)



In [7]:
def train(args):
    rank, world_size, local_rank, backend = maybe_init_dist()

    # Device policy: only use GPU when NCCL backend is active (Linux multi-GPU). Otherwise use CPU.
    use_gpu = backend == "nccl"
    device = torch.device(f"cuda:{local_rank}" if use_gpu else "cpu")

    # Data
    train_loader, test_loader, train_sampler = build_loaders(
        batch_size=args.batch_size,
        world_size=world_size,
        rank=rank,
        num_workers=args.workers,
    )

    # Model & optimizer
    model = MLP().to(device)
    if world_size > 1:
        ddp_model = DDP(model, device_ids=[local_rank] if use_gpu else None)
    else:
        ddp_model = model

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=args.lr * max(1, world_size), momentum=0.9)
    scaler = torch.amp.GradScaler(device="cuda") if (args.amp and use_gpu) else None

    epoch_times, comm_times = [], []

    for epoch in range(args.epochs):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        ddp_model.train()

        epoch_start = time.time()
        comm_time = 0.0

        optimizer.zero_grad(set_to_none=True)

        for step, (x, y) in enumerate(train_loader, start=1):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # Optional gradient accumulation
            maybe_no_sync = (
                (world_size > 1) and (args.grad_accum_steps > 1) and (step % args.grad_accum_steps != 0)
            )

            ctx = ddp_model.no_sync() if maybe_no_sync else torch.no_grad() if False else None
            # Simple context manager shim
            class _NullCtx:
                def __enter__(self):
                    return None
                def __exit__(self, exc_type, exc, tb):
                    return False
            if ctx is None:
                ctx = _NullCtx()

            step_start = time.time()
            with ctx:
                if scaler is not None:
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        logits = ddp_model(x)
                        loss = criterion(logits, y) / args.grad_accum_steps
                    scaler.scale(loss).backward()
                else:
                    logits = ddp_model(x)
                    loss = criterion(logits, y) / args.grad_accum_steps
                    loss.backward()

            # Communication happens during backward on synced steps; we approximate comm time as time after backward
            if use_gpu:
                torch.cuda.synchronize()
            after_backward = time.time()

            do_step = (step % args.grad_accum_steps == 0)
            if do_step:
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            if use_gpu:
                torch.cuda.synchronize()
            after_step = time.time()

            comm_time += max(0.0, after_step - after_backward)

            if rank == 0 and args.log_interval and step % args.log_interval == 0:
                print(f"epoch {epoch} step {step}: loss={loss.item()*args.grad_accum_steps:.4f}")

        epoch_end = time.time()
        epoch_times.append(epoch_end - epoch_start)
        comm_times.append(comm_time)

        # Evaluation on rank 0
        if rank == 0:
            acc = evaluate(ddp_model if world_size == 1 else model, test_loader, device)
            print(
                f"Epoch {epoch}: epoch_time={epoch_times[-1]:.3f}s, comm_time_est={comm_times[-1]:.3f}s, test_acc={acc:.2f}%"
            )

    if rank == 0:
        print("Per-epoch times:", [round(t, 3) for t in epoch_times])
        print("Estimated comm times:", [round(t, 3) for t in comm_times])

    if dist.is_initialized():
        dist.destroy_process_group()

In [8]:
if __name__ == "__main__":
    class Args:
        def __init__(self):
            self.epochs=20
            self.batch_size=64
            self.workers=4
            self.lr=0.01
            self.amp="store_action"
            self.grad_accum_steps=1
            self.log_interval=200
    args = Args()
    train(args)

epoch 0 step 200: loss=0.2910
epoch 0 step 400: loss=0.1391
epoch 0 step 600: loss=0.2128
epoch 0 step 800: loss=0.0632
Epoch 0: epoch_time=11.177s, comm_time_est=1.409s, test_acc=96.18%
epoch 1 step 200: loss=0.0679
epoch 1 step 400: loss=0.0412
epoch 1 step 600: loss=0.1900
epoch 1 step 800: loss=0.0847
Epoch 1: epoch_time=7.805s, comm_time_est=1.464s, test_acc=97.36%
epoch 2 step 200: loss=0.1713
epoch 2 step 400: loss=0.2485
epoch 2 step 600: loss=0.0235
epoch 2 step 800: loss=0.0099
Epoch 2: epoch_time=7.891s, comm_time_est=1.481s, test_acc=97.55%
epoch 3 step 200: loss=0.0976
epoch 3 step 400: loss=0.0898
epoch 3 step 600: loss=0.1281
epoch 3 step 800: loss=0.0746
Epoch 3: epoch_time=7.341s, comm_time_est=1.457s, test_acc=97.79%
epoch 4 step 200: loss=0.1347
epoch 4 step 400: loss=0.0127
epoch 4 step 600: loss=0.0135
epoch 4 step 800: loss=0.0426
Epoch 4: epoch_time=7.375s, comm_time_est=1.509s, test_acc=98.07%
epoch 5 step 200: loss=0.0264
epoch 5 step 400: loss=0.0224
epoch 5 s